# Colab Session: paddleocr-ec146
Generated from colab-cli history log.

**Session Created**: 2026-07-29 20:41:38
- Endpoint: `gpu-t4-s-kkb-usw1b0-xgcrdqm6sw3e`
- Hardware: `T4`

In [ ]:
"""Install the current PaddleOCR GPU stack in Google Colab with uv."""

from __future__ import annotations

import shutil
import subprocess

UV = shutil.which("uv")
if UV is None:
    raise RuntimeError("uv is required; current Colab runtimes include it.")

subprocess.run(
    [
        UV,
        "pip",
        "install",
        "--system",
        "--index-url",
        "https://www.paddlepaddle.org.cn/packages/stable/cu126/",
        "paddlepaddle-gpu==3.3.0",
    ],
    check=True,
)
subprocess.run(
    [
        UV,
        "pip",
        "install",
        "--system",
        "paddleocr>=3.7,<4",
    ],
    check=True,
)

import paddle  # noqa: E402
import paddleocr  # noqa: E402

print(f"paddle={paddle.__version__}")
print(f"paddleocr={paddleocr.__version__}")
print(f"compiled_with_cuda={paddle.device.is_compiled_with_cuda()}")
print(f"device={paddle.device.get_device()}")



/usr/local/lib/python3.12/dist-packages/paddle/utils/cpp_extension/extension_utils.py:712: UserWarning: No ccache found. Please be aware that recompiling all source files may be required. You can download and install ccache from: https://github.com/ccache/ccache/blob/master/doc/INSTALL.md
  warnings.warn(warning_message)


ImportError: /usr/local/lib/python3.12/dist-packages/torch/lib/libtorch_cuda.so: undefined symbol: ncclCommShrink

In [ ]:
"""Install the current PaddleOCR GPU stack in Google Colab with uv."""

from __future__ import annotations

import shutil
import subprocess

UV = shutil.which("uv")
if UV is None:
    raise RuntimeError("uv is required; current Colab runtimes include it.")

subprocess.run(
    [
        UV,
        "pip",
        "install",
        "--system",
        "--index-url",
        "https://www.paddlepaddle.org.cn/packages/stable/cu126/",
        "paddlepaddle-gpu==3.3.0",
    ],
    check=True,
)
subprocess.run(
    [
        UV,
        "pip",
        "install",
        "--system",
        "paddleocr>=3.7,<4",
    ],
    check=True,
)

# Paddle's CUDA wheel can replace NCCL with a version incompatible with the
# PyTorch preinstalled by Colab. PaddleOCR does not need PyTorch for its native
# engine, but an optional ModelScope probe imports it when present.
subprocess.run(
    [
        UV,
        "pip",
        "uninstall",
        "--system",
        "torch",
        "torchvision",
        "torchaudio",
    ],
    check=False,
)

print("PaddleOCR GPU stack installed. Restart the kernel before importing it.")


PaddleOCR GPU stack installed. Restart the kernel before importing it.


*File Operation*: `upload` on `/content/sapl-emenda_146.pdf`

In [ ]:
"""Benchmark PaddleOCR on the EC 146 PDF and export Markdown plus metrics."""

from __future__ import annotations

import json
import statistics
import time
from pathlib import Path
from typing import Any

import paddle
from paddleocr import PaddleOCR

INPUT_PATH = Path("/content/sapl-emenda_146.pdf")
OUTPUT_DIR = Path("/content/output/paddleocr-ec146")
MARKDOWN_PATH = OUTPUT_DIR / "sapl-emenda_146-paddleocr.md"
METRICS_PATH = OUTPUT_DIR / "metrics.json"


def result_payload(result: Any) -> dict[str, Any]:
    payload = result.json
    if callable(payload):
        payload = payload()
    return payload.get("res", payload)


def run_pass(ocr: PaddleOCR, *, save_output: bool) -> tuple[float, list[float], list[str]]:
    started = time.perf_counter()
    last_page_finished = started
    page_times: list[float] = []
    markdown_pages: list[str] = []

    for page_number, result in enumerate(ocr.predict_iter(str(INPUT_PATH)), start=1):
        now = time.perf_counter()
        page_times.append(now - last_page_finished)
        last_page_finished = now

        payload = result_payload(result)
        texts = [text.strip() for text in payload.get("rec_texts", []) if text.strip()]
        scores = [float(score) for score in payload.get("rec_scores", [])]
        mean_score = statistics.fmean(scores) if scores else 0.0
        markdown_pages.append(
            f"<!-- Página {page_number}; confiança média {mean_score:.4f} -->\n\n"
            + "\n\n".join(texts)
        )

        if save_output:
            result.save_to_json(str(OUTPUT_DIR / "json"))

        print(
            f"Page {page_number}: {page_times[-1]:.3f}s, "
            f"{len(texts)} lines, mean confidence {mean_score:.4f}",
            flush=True,
        )

    return time.perf_counter() - started, page_times, markdown_pages


def main() -> None:
    if not INPUT_PATH.is_file():
        raise FileNotFoundError(INPUT_PATH)
    if not paddle.device.is_compiled_with_cuda():
        raise RuntimeError("The installed PaddlePaddle build has no CUDA support.")

    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    paddle.set_device("gpu:0")

    initialization_started = time.perf_counter()
    ocr = PaddleOCR(
        lang="pt",
        ocr_version="PP-OCRv6",
        use_doc_orientation_classify=False,
        use_doc_unwarping=False,
        use_textline_orientation=False,
        device="gpu:0",
    )
    initialization_time = time.perf_counter() - initialization_started
    print(f"Pipeline initialization: {initialization_time:.3f}s", flush=True)

    cold_total, cold_pages, markdown_pages = run_pass(ocr, save_output=True)
    MARKDOWN_PATH.write_text(
        "\n\n".join(markdown_pages).strip() + "\n",
        encoding="utf-8",
    )

    warm_total, warm_pages, _ = run_pass(ocr, save_output=False)
    metrics = {
        "gpu": paddle.device.cuda.get_device_name(0),
        "paddle": paddle.__version__,
        "input": str(INPUT_PATH),
        "pages": len(cold_pages),
        "pipeline_initialization_seconds": initialization_time,
        "cold_inference_seconds": cold_total,
        "cold_page_seconds": cold_pages,
        "warm_inference_seconds": warm_total,
        "warm_page_seconds": warm_pages,
        "cold_mean_seconds_per_page": statistics.fmean(cold_pages),
        "warm_mean_seconds_per_page": statistics.fmean(warm_pages),
    }
    METRICS_PATH.write_text(
        json.dumps(metrics, indent=2, ensure_ascii=False) + "\n",
        encoding="utf-8",
    )
    print(json.dumps(metrics, indent=2, ensure_ascii=False), flush=True)
    print(f"Markdown: {MARKDOWN_PATH}", flush=True)


if __name__ == "__main__":
    main()


/usr/local/lib/python3.12/dist-packages/paddle/utils/cpp_extension/extension_utils.py:712: UserWarning: No ccache found. Please be aware that recompiling all source files may be required. You can download and install ccache from: https://github.com/ccache/ccache/blob/master/doc/INSTALL.md
  warnings.warn(warning_message)


[transformers] PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


Creating model: ('PP-OCRv6_medium_det', None, None)


Checking connectivity to the model hosters, this may take a while. To bypass this check, set `PADDLE_PDX_DISABLE_MODEL_SOURCE_CHECK` to `True`.


Using official model (PP-OCRv6_medium_det), the model files will be automatically downloaded and saved in `/root/.paddlex/official_models/PP-OCRv6_medium_det`.


...

Page 8: 1.614s, 40 lines, mean confidence 0.9870


Page 9: 1.574s, 41 lines, mean confidence 0.9715


Page 10: 0.860s, 26 lines, mean confidence 0.9854


{
  "gpu": "Tesla T4",
  "paddle": "3.3.0",
  "input": "/content/sapl-emenda_146.pdf",
  "pages": 10,
  "pipeline_initialization_seconds": 19.289637710999955,
  "cold_inference_seconds": 15.952748625000027,
  "cold_page_seconds": [
    2.3059831699999904,
    1.2400606630000084,
    1.82639170799996,
    1.5675397280000425,
    1.5743120729999873,
    1.9412502510000422,
    1.450407326000004,
    1.5916286529999297,
    1.5677250290000302,
    0.882908010000051
  ],
  "warm_inference_seconds": 14.82587966799997,
  "warm_page_seconds": [
    1.214028606999932,
    1.2028982990000259,
    1.8007513600000493,
    1.682346761999952,
    1.6232121670000197,
    1.7851947399999517,
    1.4663078859999814,
    1.6137624360000018,
    1.574493817000075,
    0.8603515349999498
  ],
  "cold_mean_seconds_per_page": 1.5948206611000046,
  "warm_mean_seconds_per_page": 1.482334760899994
}


Markdown: /content/output/paddleocr-ec146/sapl-emenda_146-paddleocr.md


*File Operation*: `download` on `/content/output/paddleocr-ec146/sapl-emenda_146-paddleocr.md`

*File Operation*: `download` on `/content/output/paddleocr-ec146/metrics.json`